In [2]:
import pandas as pd
import yfinance as yf
import numpy as np
from datetime import date

stocks = [
    "ADANIENT.NS",  "ADANIPORTS.NS", "APOLLOHOSP.NS", "ASIANPAINT.NS",
    "AXISBANK.NS",  "BAJAJ-AUTO.NS", "BAJAJFINSV.NS", "BAJFINANCE.NS",
    "BEL.NS",       "BHARTIARTL.NS", "BPCL.NS",       "BRITANNIA.NS",
    "CIPLA.NS",     "COALINDIA.NS",  "DIVISLAB.NS",   "DRREDDY.NS",
    "EICHERMOT.NS", "GRASIM.NS",     "HCLTECH.NS",    "HDFCBANK.NS",
    "HDFCLIFE.NS",  "HEROMOTOCO.NS", "HINDALCO.NS",   "HINDUNILVR.NS",
    "ICICIBANK.NS", "INDUSINDBK.NS", "INFY.NS",       "ITC.NS",
    "JIOFIN.NS",    "JSWSTEEL.NS",   "KOTAKBANK.NS",  "LT.NS",
    "M&M.NS",       "MARUTI.NS",     "NESTLEIND.NS",  "NTPC.NS",
    "ONGC.NS",      "POWERGRID.NS",  "RELIANCE.NS",   "SBILIFE.NS",
    "SBIN.NS",      "SHRIRAMFIN.NS", "SUNPHARMA.NS",  "TATACONSUM.NS",
    "TATASTEEL.NS",  "TCS.NS",        "TECHM.NS",
    "TITAN.NS",     "ULTRACEMCO.NS"
]

end = pd.to_datetime(date.today())
start = end - pd.DateOffset(years=1)
mom_start = end - pd.DateOffset(years=2)

price = yf.download(tickers=stocks, start=mom_start, end=end)["Close"]

C:\Users\preet\AppData\Local\Temp\ipykernel_11440\1182104832.py:26: FutureWarning: YF.download() has changed argument auto_adjust default to True
  price = yf.download(tickers=stocks, start=mom_start, end=end)["Close"]
[*********************100%***********************]  49 of 49 completed


In [3]:
price["ADANIENT.NS"][price.index <= start][-1]

C:\Users\preet\AppData\Local\Temp\ipykernel_11440\3397428422.py:1: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  price["ADANIENT.NS"][price.index <= start][-1]


np.float64(2334.093017578125)

In [4]:
mcap = pd.DataFrame({
    stock : price[stock][price.index <= start].iloc[-1] * yf.Ticker(stock).info["sharesOutstanding"]
    for stock in price.columns
},index=["mcap"]).T
mcap["rank"] = mcap["mcap"].rank()

size_stk = {}
for i in range(len(mcap)):
    if mcap.iloc[i,1] <= 10:
        size_stk[mcap.index[i]] = "Long"
    elif mcap.iloc[i,1] > len(mcap) - 10:
        size_stk[mcap.index[i]] ="Short"
    else:
        continue

size_p = pd.DataFrame(size_stk, index=["Positions"]).T
size_p = size_p.sort_values("Positions")
size_p = size_p.reset_index()
size_p.rename(columns = {"index":"Stock"}, inplace = True)

size_p[f"Price as on: {start.date()}"] = size_p["Stock"].map(price.asof(start))
size_p[f"Price as on: {end}"] = size_p["Stock"].map(price.asof(pd.to_datetime(end)))

for i in range(len(size_p)):
    if size_p.loc[i,"Positions"] == "Long":
        size_p.loc[i,"Return"] = size_p.loc[i,f"Price as on: {end}"] / size_p.loc[i,f"Price as on: {start.date()}"] - 1
    elif size_p.loc[i,"Positions"] == "Short":
        size_p.loc[i,"Return"] = 1- size_p.loc[i,f"Price as on: {end}"] / size_p.loc[i,f"Price as on: {start.date()}"]
    else:
        continue

size_short_return = np.average(size_p["Return"][size_p["Positions"]=="Short"])
size_long_return = np.average(size_p["Return"][size_p["Positions"]=="Long"])
size_portfolio_return = np.average(size_p["Return"])



In [5]:
pb = pd.DataFrame({
    stock : price[stock][price.index <= start].iloc[-1] / yf.Ticker(stock).info["bookValue"]
    for stock in price.columns
}, index=["p/b"]).T
pb["rank"] = pb["p/b"].rank()

value_stk = {}
for i in range(len(pb)):
    if pb.iloc[i,1] <= 10:
        value_stk[pb.index[i]] = "Long"
    elif pb.iloc[i,1] > len(pb) - 10:
        value_stk[pb.index[i]] = "Short"
    else:
        continue

value_p = pd.DataFrame(value_stk, index=["Positions"]).T
value_p = value_p.sort_values("Positions")
value_p = value_p.reset_index()
value_p.rename(columns = {"index":"Stock"}, inplace = True)

value_p[f"Price as on: {start.date()}"] = value_p["Stock"].map(price.asof(start))
value_p[f"Price as on: {end}"] = value_p["Stock"].map(price.asof(pd.to_datetime(end)))

for i in range(len(value_p)):
    if value_p.loc[i,"Positions"] == "Long":
        value_p.loc[i,"Return"] = value_p.loc[i,f"Price as on: {end}"] / value_p.loc[i,f"Price as on: {start.date()}"] - 1
    elif value_p.loc[i,"Positions"] == "Short":
        value_p.loc[i,"Return"] = 1- value_p.loc[i,f"Price as on: {end}"] / value_p.loc[i,f"Price as on: {start.date()}"]
    else:
        continue

value_short_return = np.average(value_p["Return"][value_p["Positions"]=="Short"])
value_long_return = np.average(value_p["Return"][value_p["Positions"]=="Long"])
value_portfolio_return = np.average(value_p["Return"])


In [6]:
price["ADANIENT.NS"][price.index >= mom_start][0]

C:\Users\preet\AppData\Local\Temp\ipykernel_11440\1071341588.py:1: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  price["ADANIENT.NS"][price.index >= mom_start][0]


np.float64(3249.21826171875)

In [7]:
mom = pd.DataFrame({
    stock: price[stock][price.index >= mom_start].iloc[0] / price[stock][price.index >= start].iloc[0] -1
    for stock in price.columns
}, index = ["mom_score"]).T
mom["rank"] = mom["mom_score"].rank(ascending = False)

mom_stk = {}
for i in range(len(mom)):
    if mom.iloc[i,1] <= 10:
        mom_stk[mom.index[i]] = "Long"
    elif mom.iloc[i,1] > len(pb) - 10:
        mom_stk[mom.index[i]] = "Short"
    else:
        continue

mom_p = pd.DataFrame(mom_stk, index=["Positions"]).T
mom_p = mom_p.sort_values("Positions")
mom_p = mom_p.reset_index()
mom_p.rename(columns = {"index":"Stock"}, inplace = True)

mom_p[f"Price as on: {start.date()}"] = mom_p["Stock"].map(price.asof(start))
mom_p[f"Price as on: {end}"] = mom_p["Stock"].map(price.asof(pd.to_datetime(end)))

for i in range(len(mom_p)):
    if mom_p.loc[i,"Positions"] == "Long":
        mom_p.loc[i,"Return"] = mom_p.loc[i,f"Price as on: {end}"] / mom_p.loc[i,f"Price as on: {start.date()}"] - 1
    elif mom_p.loc[i,"Positions"] == "Short":
        mom_p.loc[i,"Return"] = 1- mom_p.loc[i,f"Price as on: {end}"] / mom_p.loc[i,f"Price as on: {start.date()}"]
    else:
        continue

mom_short_return = np.average(mom_p["Return"][mom_p["Positions"]=="Short"])
mom_long_return = np.average(mom_p["Return"][mom_p["Positions"]=="Long"])
mom_portfolio_return = np.average(mom_p["Return"])


In [8]:
print(size_long_return, size_short_return, size_portfolio_return)
print(value_long_return, value_short_return, value_portfolio_return)
print(mom_long_return, mom_short_return, mom_portfolio_return)

0.11758450141741009 0.06946499989356832 0.09352475065548922
0.11829357688965894 -0.051292444836240736 0.0335005660267091
0.08827341346260383 -0.10220591357746842 -0.0069662500574323


In [9]:
def sharpe_ratio(portfolio_return,risk_free,m_returns):
    sharpe = (portfolio_return - risk_free)/np.std(m_returns)
    return sharpe

r_f = .065

In [10]:
monthly_price = price.resample("ME").last()
monthly_returns = monthly_price[start:end].pct_change()

monthly_returns.head(3)

Ticker,ADANIENT.NS,ADANIPORTS.NS,APOLLOHOSP.NS,ASIANPAINT.NS,AXISBANK.NS,BAJAJ-AUTO.NS,BAJAJFINSV.NS,BAJFINANCE.NS,BEL.NS,BHARTIARTL.NS,...,SBILIFE.NS,SBIN.NS,SHRIRAMFIN.NS,SUNPHARMA.NS,TATACONSUM.NS,TATASTEEL.NS,TCS.NS,TECHM.NS,TITAN.NS,ULTRACEMCO.NS
Date,,,,,,,,,,,,,,,,,,,,,
2025-04-30,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2025-05-31,0.094990,0.177805,-0.013831,-0.068681,0.006076,0.071856,0.033716,0.069795,0.224451,-0.004452,...,0.026277,0.05067,0.045202,-0.084429,-0.044005,0.149486,0.002809,0.047172,0.051869,-0.037024
2025-06-30,0.040001,0.017068,0.052540,0.045859,0.005871,-0.002175,0.019630,0.020097,0.095944,0.082642,...,0.014402,0.00991,0.105576,-0.001133,-0.006689,0.015313,0.008480,0.071860,0.038031,0.078769


In [ ]:
size_m_ret = pd.DataFrame({
    stock : monthly_returns[stock]
    for stock in size_stk.keys()
},  index=monthly_returns.index, columns=size_stk.keys())

size_m_ret["return"] = monthly_returns[size_p["Stock"][size_p["Positions"]=="Long"]].mean(axis=1) -monthly_returns[size_p["Stock"][size_p["Positions"]=="Short"]].mean(axis=1)


value_m_ret = pd.DataFrame({
    stock : monthly_returns[stock]
    for stock in value_stk.keys()
},  index=monthly_returns.index, columns=value_stk.keys())

value_m_ret["return"] = monthly_returns[value_p["Stock"][value_p["Positions"]=="Long"]].mean(axis=1) -monthly_returns[value_p["Stock"][value_p["Positions"]=="Short"]].mean(axis=1)


mom_m_ret = pd.DataFrame({
    stock : monthly_returns[stock]
    for stock in mom_stk.keys()
},  index=monthly_returns.index, columns=mom_stk.keys())

mom_m_ret["return"] = monthly_returns[mom_p["Stock"][mom_p["Positions"]=="Long"]].mean(axis=1) -monthly_returns[mom_p["Stock"][mom_p["Positions"]=="Short"]].mean(axis=1)


In [18]:

size_sharpe = sharpe_ratio(risk_free=r_f, portfolio_return=size_portfolio_return,m_returns=size_m_ret["return"])
value_sharpe = sharpe_ratio(risk_free=r_f, portfolio_return=size_portfolio_return,m_returns=value_m_ret["return"])
mom_sharpe = sharpe_ratio(risk_free=r_f, portfolio_return=size_portfolio_return,m_returns=mom_m_ret["return"])

sharpe_ratios = pd.DataFrame({
    "size" : size_sharpe,
    "value" : value_sharpe,
    "mom" : mom_sharpe
}, index=["Sharpe Ratio"]).T

sharpe_ratios

,Sharpe Ratio
size,1.629250
value,0.992766
mom,1.208532
